# 05 — آمار توصیفی و تحلیل زمانی

این Notebook فازهای نهم و دهم checklist.md (§21 و §22) را روایت می‌کند.  
منطق آماری در `src/temporal_analysis/descriptive_stats.py` و `src/temporal_analysis/weekly_trend.py` پیاده شده و اینجا فقط فراخوانی و تفسیر نتایج انجام می‌شود.

**ترتیب اجرا:** پس از `04_full_annotation.ipynb` — نیازمند `data/processed/annotated_dataset.parquet` واقعی.

---
⚠️ **هشدار داده:** نتایج فعلاً روی `data/processed/annotated_dataset.sample.parquet` (fallback مصنوعی طبق `docs/pipeline_b_input_contract.md`) هستن، نه dataset واقعی Pipeline A. اعداد زیر تا رسیدن داده واقعی **جنبه تفسیری ندارند**.

---

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=None):
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "config" / "config.yaml").exists() or (candidate / "src" / "temporal_analysis").exists():
            return candidate
    raise FileNotFoundError("Project root not found — run from within the media-sentiment-pipeline repo.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT = {ROOT}")

In [ ]:
import pandas as pd

REAL_INPUT    = ROOT / "data" / "processed" / "annotated_dataset.parquet"
SAMPLE_INPUT  = ROOT / "data" / "processed" / "annotated_dataset.sample.parquet"
OUTPUT_DIR    = ROOT / "outputs" / "tables"

real_data_ready = REAL_INPUT.exists() and REAL_INPUT.stat().st_size > 0
sample_ready    = SAMPLE_INPUT.exists() and SAMPLE_INPUT.stat().st_size > 0

print({
    "real_annotated_dataset_exists": real_data_ready,
    "sample_fallback_exists": sample_ready,
    "active_input": str(REAL_INPUT) if real_data_ready else (str(SAMPLE_INPUT) if sample_ready else "NONE"),
})

---
## بخش ۱ — آمار توصیفی (§21)

---
⚠️ **هشدار داده:** نتایج فعلاً روی `data/processed/annotated_dataset.sample.parquet` (fallback مصنوعی طبق `docs/pipeline_b_input_contract.md`) هستن، نه dataset واقعی Pipeline A.

---

### قرارداد آماری — §21

- واحد تحلیل: هر ردیف `annotated_dataset.parquet` یک محتوای واجد شرایط (eligible) است.
- آمار توصیفی در چهار سطح محاسبه می‌شود: **کل، به‌تفکیک پلتفرم، به‌تفکیک هفته (pooled‐supplementary)، به‌تفکیک (پلتفرم × هفته)** — آخری سطح اصلی است.
- سهم‌های برچسب‌دار (sentiment/stance/emotion/content_type) فقط از ردیف‌های `annotation_status == "ok"` محاسبه می‌شود (قانون §24).
- سهم‌های زبان/منبع/automation-risk از **همه** ردیف‌ها محاسبه می‌شود — حذف آن‌ها داده را دست‌کم‌شمار می‌کند.
- نرخ حذف annotation (`pct_non_ok`) به‌صراحت گزارش می‌شود.
- X، Reddit و YouTube **جداگانه** تحلیل می‌شوند؛ اعداد تجمیع‌یافته فقط مکمل هستند.

In [ ]:
# بررسی وجود خروجی‌های از پیش محاسبه‌شده
DESC_FILES = {
    "overall":          OUTPUT_DIR / "descriptive_stats_overall.csv",
    "by_platform":      OUTPUT_DIR / "descriptive_stats_by_platform.csv",
    "by_platform_week": OUTPUT_DIR / "descriptive_stats_by_platform_week.csv",
    "category_shares":  OUTPUT_DIR / "descriptive_stats_category_shares.csv",
    "missing_rates":    OUTPUT_DIR / "descriptive_stats_missing_rates.csv",
    "annotation_coverage": OUTPUT_DIR / "descriptive_stats_annotation_coverage.csv",
}

desc_outputs_exist = all(p.exists() for p in DESC_FILES.values())
print(f"خروجی‌های descriptive_stats موجود: {desc_outputs_exist}")

In [ ]:
if desc_outputs_exist:
    # بارگذاری خروجی‌های از پیش محاسبه‌شده
    desc_overall    = pd.read_csv(DESC_FILES["overall"])
    desc_platform   = pd.read_csv(DESC_FILES["by_platform"])
    desc_plat_week  = pd.read_csv(DESC_FILES["by_platform_week"])
    category_shares = pd.read_csv(DESC_FILES["category_shares"])
    ann_coverage    = pd.read_csv(DESC_FILES["annotation_coverage"])
    print("STATUS: loaded_from_precomputed_csvs")
elif real_data_ready or sample_ready:
    # اجرای مستقیم descriptive_stats.py
    from src.temporal_analysis.descriptive_stats import build_tables
    from src.temporal_analysis.common import load_annotated_dataset

    input_path = REAL_INPUT if real_data_ready else SAMPLE_INPUT
    print(f"در حال محاسبه از: {input_path.name} ...")

    df = load_annotated_dataset(input_path)
    tables = build_tables(df)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, tbl in tables.items():
        tbl.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

    desc_overall    = tables["descriptive_stats_overall"]
    desc_platform   = tables["descriptive_stats_by_platform"]
    desc_plat_week  = tables["descriptive_stats_by_platform_week"]
    category_shares = tables["descriptive_stats_category_shares"]
    ann_coverage    = tables["descriptive_stats_annotation_coverage"]
    print(f"STATUS: computed — {len(df):,} ردیف")
else:
    print("STATUS: pending_annotated_dataset")
    print(f"ورودی مورد انتظار: {REAL_INPUT}")
    desc_overall = desc_platform = desc_plat_week = category_shares = ann_coverage = None

### ۱.۱ — آمار کلی مجموعه داده

In [ ]:
if desc_overall is not None:
    display(desc_overall)
else:
    print("STATUS: pending_annotated_dataset")

### ۱.۲ — آمار به‌تفکیک پلتفرم (X / Reddit / YouTube جدا)

In [ ]:
if desc_platform is not None:
    display(desc_platform)
else:
    print("STATUS: pending_annotated_dataset")

### ۱.۳ — پوشش annotation و نرخ حذف (§24)

In [ ]:
if ann_coverage is not None:
    plat_level = ann_coverage[ann_coverage["group_level"] == "platform"]
    display(plat_level[["platform", "n_total", "n_ok", "n_non_ok", "pct_non_ok"]])
else:
    print("STATUS: pending_annotated_dataset")

### ۱.۴ — سهم‌های دسته‌بندی به‌تفکیک پلتفرم

زبان، منبع، automation risk، و چهار محور برچسب (`sentiment/stance/emotion/content_type`) — برچسب‌ها فقط از ردیف‌های `annotation_status=="ok"` محاسبه شده.

In [ ]:
if category_shares is not None:
    plat_shares = category_shares[category_shares["group_level"] == "platform"]
    for platform, grp in plat_shares.groupby("platform"):
        print(f"\n=== {platform.upper()} ===")
        display(grp[["dimension", "category", "n", "proportion"]].sort_values(["dimension", "proportion"], ascending=[True, False]))
else:
    print("STATUS: pending_annotated_dataset")

---
## بخش ۲ — روند هفتگی با Wilson CI (§22)

---
⚠️ **هشدار داده:** نتایج فعلاً روی `data/processed/annotated_dataset.sample.parquet` (fallback مصنوعی طبق `docs/pipeline_b_input_contract.md`) هستن، نه dataset واقعی Pipeline A.

---

### قرارداد آماری — §22

- **مهم‌ترین اصل:** X، Reddit و YouTube **جداگانه** تحلیل می‌شوند — هیچ عدد pooled در تحلیل اصلی وجود ندارد.
- فاصله اطمینان: **Wilson Score 95%** — بدون scipy، بر اساس فرمول بسته (همان‌طور که `src/temporal_analysis/common.py` پیاده کرده).
- هفته‌های با `n < 30`: علامت `is_low_sample=True` — **حذف نمی‌شوند**، فقط توصیفی گزارش می‌شوند.
- هفته W21 (جزئی، ۵ روز): علامت `is_partial_week=True` — حجم آن با هفته‌های کامل مقایسه نمی‌شود.
- هفته‌های بدون داده: `is_data_gap=True`، مقدار NaN — **هیچ هفته‌ای به‌صورت silent حذف نمی‌شود**.
- ردیف‌های `dataset_target=="opinion_untimed"` (بدون `project_week`) از این بخش حذف‌اند و فقط در §21 گزارش می‌شوند.
- چهار محور: `sentiment`, `stance`, `emotion`, `content_type` — هر کلاس کانونیک حتی با count صفر ردیف دارد.

In [ ]:
TREND_FILES = {
    axis: OUTPUT_DIR / f"weekly_trend_{axis}_by_platform.csv"
    for axis in ["sentiment", "stance", "emotion", "content_type"]
}

trend_outputs_exist = all(p.exists() for p in TREND_FILES.values())
print(f"خروجی‌های weekly_trend موجود: {trend_outputs_exist}")

In [ ]:
if trend_outputs_exist:
    trend_tables = {axis: pd.read_csv(path) for axis, path in TREND_FILES.items()}
    print("STATUS: loaded_from_precomputed_csvs")
elif real_data_ready or sample_ready:
    from src.temporal_analysis.weekly_trend import build_all_trends
    from src.temporal_analysis.common import load_annotated_dataset

    # اگر df از بخش ۱ در حافظه است، دوباره لود نمی‌کنیم
    if 'df' not in dir():
        input_path = REAL_INPUT if real_data_ready else SAMPLE_INPUT
        from src.temporal_analysis.common import load_annotated_dataset
        df = load_annotated_dataset(input_path)

    trend_tables = build_all_trends(df)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, tbl in trend_tables.items():
        tbl.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"STATUS: computed — {len(df):,} ردیف")
else:
    print("STATUS: pending_annotated_dataset")
    trend_tables = {}

### ۲.۱ — روند هفتگی Sentiment (به‌تفکیک پلتفرم)

In [ ]:
if "sentiment" in trend_tables:
    t = trend_tables["sentiment"]
    for platform, grp in t.groupby("platform"):
        print(f"\n=== {platform.upper()} — Sentiment ===")
        summary = (
            grp[["project_week", "class_label", "n", "class_proportion",
                 "wilson_ci_low", "wilson_ci_high", "is_low_sample",
                 "is_partial_week", "is_data_gap"]]
            .sort_values(["project_week", "class_label"])
        )
        display(summary)
else:
    print("STATUS: pending_annotated_dataset")

### ۲.۲ — روند هفتگی Stance (به‌تفکیک پلتفرم)

In [ ]:
if "stance" in trend_tables:
    t = trend_tables["stance"]
    for platform, grp in t.groupby("platform"):
        print(f"\n=== {platform.upper()} — Stance ===")
        display(
            grp[["project_week", "class_label", "n", "class_proportion",
                 "wilson_ci_low", "wilson_ci_high", "is_low_sample",
                 "is_partial_week", "is_data_gap"]]
            .sort_values(["project_week", "class_label"])
        )
else:
    print("STATUS: pending_annotated_dataset")

### ۲.۳ — روند هفتگی Emotion (به‌تفکیک پلتفرم)

In [ ]:
if "emotion" in trend_tables:
    t = trend_tables["emotion"]
    for platform, grp in t.groupby("platform"):
        print(f"\n=== {platform.upper()} — Emotion ===")
        display(
            grp[["project_week", "class_label", "n", "class_proportion",
                 "wilson_ci_low", "wilson_ci_high", "is_low_sample",
                 "is_partial_week", "is_data_gap"]]
            .sort_values(["project_week", "class_label"])
        )
else:
    print("STATUS: pending_annotated_dataset")

### ۲.۴ — روند هفتگی Content Type (به‌تفکیک پلتفرم)

In [ ]:
if "content_type" in trend_tables:
    t = trend_tables["content_type"]
    for platform, grp in t.groupby("platform"):
        print(f"\n=== {platform.upper()} — Content Type ===")
        display(
            grp[["project_week", "class_label", "n", "class_proportion",
                 "wilson_ci_low", "wilson_ci_high", "is_low_sample",
                 "is_partial_week", "is_data_gap"]]
            .sort_values(["project_week", "class_label"])
        )
else:
    print("STATUS: pending_annotated_dataset")

### یادداشت تفسیری §22

- هفته‌های با `is_low_sample=True` (n < 30): Wilson CI پهن‌تر است — اعداد را ببینید، نه نتیجه‌گیری نکنید.
- هفته W21 (`is_partial_week=True`): فقط ۵ روز — حجم آن به هفته‌های کامل مقایسه نمی‌شود.
- هفته‌های `is_data_gap=True`: ردیف وجود دارد تا gap قابل مشاهده باشد، نه اینکه پنهان بماند.
- هیچ pooling سه‌پلتفرمی در تحلیل اصلی انجام نمی‌شود.

---
## بخش ۳ — Composition Shift (§23) — TODO

> **این بخش در حال ساخت است.**
>
> تحلیل Composition Shift (§23 checklist) داره موازی توسط **پارمیدا** در `src/temporal_analysis/` ساخته می‌شه. بعد از اتمام، نتایج آن به همین بخش append می‌شه — ساختار نوت‌بوک طراحی شده که اضافه کردن آن نیازی به بازنویسی بخش‌های قبل نداشته باشد.
>
> **پرونده‌های مرتبط (از قبل موجود در `outputs/tables/`):**
> - `composition_shift_sentiment_adjusted.csv`
> - `composition_shift_platform_share.csv`
> - `composition_shift_source_share.csv`
> - `composition_shift_language_share.csv`
> - `composition_shift_content_type_share.csv`
> - `composition_shift_duplicate_share.csv`
> - `composition_shift_risk_share.csv`
> - `composition_shift_author_concentration.csv`
> - `composition_shift_parent_concentration.csv`
> - `composition_shift_query_share.csv`
>
> **نقطه وصل:** سلول‌های کد این بخش بعداً بین این سلول markdown و سلول خلاصه نهایی insert می‌شوند.

---

In [ ]:
# §23 Composition Shift — placeholder تا پیاده‌سازی پارمیدا merge شود
COMPOSITION_SHIFT_FILES = [
    OUTPUT_DIR / "composition_shift_sentiment_adjusted.csv",
    OUTPUT_DIR / "composition_shift_platform_share.csv",
    OUTPUT_DIR / "composition_shift_source_share.csv",
]

existing = [p.name for p in COMPOSITION_SHIFT_FILES if p.exists()]
print(f"Composition Shift فایل‌های موجود: {existing}")
print("STATUS: composition_shift_pending_parmida_implementation")

---
## خلاصه اجرا

| بخش | وضعیت | منبع کد |
|-----|--------|----------|
| §21 آمار توصیفی | ✅ آماده (شرطی) | `src/temporal_analysis/descriptive_stats.py` |
| §22 روند هفتگی Wilson CI | ✅ آماده (شرطی) | `src/temporal_analysis/weekly_trend.py` |
| §23 Composition Shift | ⏳ TODO — پارمیدا | `src/temporal_analysis/` |

**اجرای کامل و نهایی:** پس از رسیدن `annotated_dataset.parquet` واقعی از Pipeline A.